In [1]:
from pathlib import Path

from openicu_yaib import build_and_write_yaib_wide_for_dataset

# ---------------------------------------------------------------------------
# User settings
# ---------------------------------------------------------------------------
DATASET = "mimic-iv"
OPENICU_CONCEPT_VERSION = "1.0.0"

# Shared output directory for this project.
# The R/RICU scripts use the same directory by default.
OUTPUT_ROOT = Path.home() / "output" / "openicu_yaib"

# Optional: override these paths if your local layout differs from the defaults.
# You can also set the corresponding environment variables:
#   OPENICU_YAIB_CONCEPT_ROOT
#   OPENICU_YAIB_ICUSTAYS_CSV
#   OPENICU_YAIB_RICU_CONCEPT_DICT
CONCEPT_ROOT = None
ICUSTAYS_CSV = None
RICU_CONCEPT_DICT = None

# ---------------------------------------------------------------------------
# 1) Export all available ICU hours for downstream ML/training.
# ---------------------------------------------------------------------------
all_hours = build_and_write_yaib_wide_for_dataset(
    dataset=DATASET,
    version=OPENICU_CONCEPT_VERSION,
    max_hours=None,
    output_root=OUTPUT_ROOT,
    concept_root=CONCEPT_ROOT,
    icustays_csv=ICUSTAYS_CSV,
    ricu_concept_dict=RICU_CONCEPT_DICT,
    missing_concepts="fail",
)

print(f"Wrote all-hours OpenICU YAIB-wide parquet to: {all_hours.output_path}")
all_hours.summary


Wrote all-hours OpenICU YAIB-wide parquet to: /home/q039tl/output/openicu_yaib/openicu_dyn_all.parquet


n_rows,n_stays,min_time,max_time
u32,u32,i64,i64
8275288,94458,0,5433


In [2]:
from openicu_yaib import build_and_write_yaib_wide_for_dataset

# ---------------------------------------------------------------------------
# 2) Export the first week only for R/RICU validation.
#
# This follows the RICU/YAIB convention used in the archive comparison:
# time = 0, 1, ..., 7 * 24, i.e. max time is inclusive.
# ---------------------------------------------------------------------------
MAX_HOURS = 7 * 24

one_week = build_and_write_yaib_wide_for_dataset(
    dataset=DATASET,
    version=OPENICU_CONCEPT_VERSION,
    max_hours=MAX_HOURS,
    output_root=OUTPUT_ROOT,
    concept_root=CONCEPT_ROOT,
    icustays_csv=ICUSTAYS_CSV,
    ricu_concept_dict=RICU_CONCEPT_DICT,
    missing_concepts="fail",
)

print(f"Wrote one-week OpenICU YAIB-wide parquet to: {one_week.output_path}")
one_week.summary


Wrote one-week OpenICU YAIB-wide parquet to: /home/q039tl/output/openicu_yaib/openicu_dyn_168h.parquet


n_rows,n_stays,min_time,max_time
u32,u32,i64,i64
6267184,94458,0,168


## Optional R/RICU reference export

Run the R scripts before executing the comparison cell if you want to compare the OpenICU YAIB-wide parquet against R/RICU.

```bash
RICU_OUT_DIR="$HOME/output/openicu_yaib" Rscript scripts/export_ricu_dynamic_vars.R
RICU_OUT_DIR="$HOME/output/openicu_yaib" Rscript scripts/export_ricu_stay_windows.R
```

The comparison cell expects these files in `OUTPUT_ROOT`:

```text
ricu_dynamic_vars_miiv.parquet
ricu_stay_windows_miiv.parquet
```

The comparison uses the same window logic as the working archive notebook: RICU stay windows are converted to integer hours, capped to `MAX_HOURS`, and RICU dynamic rows are filtered to `start <= time <= end`.

`all_hours.output_path` is intended for downstream ML/training. `one_week.output_path` is only the bounded validation export.


In [3]:
from openicu_yaib import (
    compare_openicu_wide_to_ricu_for_dataset,
    display_comparison_overview,
)

# Compare the one-week OpenICU parquet against R/RICU.
# Expected valid mismatch pattern for the known issue: 14 window/stay differences.
comparison = compare_openicu_wide_to_ricu_for_dataset(
    dataset=DATASET,
    max_hours=MAX_HOURS,
    output_root=OUTPUT_ROOT,
    openicu_wide_path=one_week.output_path,
    concept_root=CONCEPT_ROOT,
    icustays_csv=ICUSTAYS_CSV,
    ricu_concept_dict=RICU_CONCEPT_DICT,
)

print(f"Wrote comparison reports to: {comparison.reports_dir}")

# The overview keeps all previous reports and additionally includes:
# - reproduction_accuracy: aggregate stay/row reproduction metrics
# - non_identical_common_stays_head: compact sample of common stays whose
#   keys or values differ (the complete report is written as CSV)
overview = display_comparison_overview(comparison)
overview


Wrote comparison reports to: /home/q039tl/output/openicu_yaib/reports/168h


{'table_summary': shape: (2, 5)
 ┌────────────────┬─────────┬─────────┬──────────┬──────────┐
 │ table          ┆ n_rows  ┆ n_stays ┆ min_time ┆ max_time │
 │ ---            ┆ ---     ┆ ---     ┆ ---      ┆ ---      │
 │ str            ┆ u32     ┆ u32     ┆ i64      ┆ i64      │
 ╞════════════════╪═════════╪═════════╪══════════╪══════════╡
 │ openicu        ┆ 6267184 ┆ 94458   ┆ 0        ┆ 168      │
 │ ricu_reference ┆ 5997455 ┆ 94448   ┆ 0        ┆ 168      │
 └────────────────┴─────────┴─────────┴──────────┴──────────┘,
 'stay_overlap': shape: (1, 5)
 ┌─────────────────┬───────────────────┬────────────────┬─────────────────────┬─────────────────────┐
 │ n_openicu_stays ┆ n_reference_stays ┆ n_common_stays ┆ n_only_openicu_stay ┆ n_only_reference_st │
 │ ---             ┆ ---               ┆ ---            ┆ s                   ┆ ays                 │
 │ i64             ┆ i64               ┆ i64            ┆ ---                 ┆ ---                 │
 │                 ┆            

### Explicit reproduction error metrics

RICU is treated as the reference. A **full row** is identical only when the `(stay_id, time)` key and every concept value match; nulls in the same positions count as equal.


In [ ]:
key_overlap = comparison.key_overlap.row(0, named=True)
stay_overlap = comparison.stay_overlap.row(0, named=True)
accuracy = comparison.reproduction_accuracy.row(0, named=True)

n_openicu_keys = key_overlap["n_openicu_keys"]
n_reference_keys = key_overlap["n_reference_keys"]
n_only_openicu_keys = key_overlap["n_only_openicu_keys"]
n_only_reference_keys = key_overlap["n_only_reference_keys"]

n_openicu_stays = stay_overlap["n_openicu_stays"]
n_reference_stays = stay_overlap["n_reference_stays"]

stay_count_error = (
    n_openicu_stays - n_reference_stays
) / n_reference_stays

key_count_error = (
    n_openicu_keys - n_reference_keys
) / n_reference_keys

key_disagreement_error = (
    n_only_openicu_keys + n_only_reference_keys
) / n_reference_keys

print("Explicit reproduction error metrics")
print("-----------------------------------")
print(
    f"Stay count error:       {stay_count_error:.6f} "
    f"({stay_count_error:.4%})"
)
print(
    f"Key count error:        {key_count_error:.6f} "
    f"({key_count_error:.4%})"
)
print(
    f"Key disagreement error: {key_disagreement_error:.6f} "
    f"({key_disagreement_error:.4%})"
)
print(
    f"Full-row error rate:    {accuracy['full_row_error_rate']:.6f} "
    f"({accuracy['full_row_error_rate']:.4%})"
)
print(
    f"Full-row match rate:    {accuracy['full_row_match_rate']:.6f} "
    f"({accuracy['full_row_match_rate']:.4%})"
)
print(
    "Full-row errors:        "
    f"{accuracy['n_full_row_errors']:,} "
    f"(only OpenICU={accuracy['n_rows_only_openicu']:,}, "
    f"only RICU={accuracy['n_rows_only_reference']:,}, "
    f"value mismatch={accuracy['n_rows_value_mismatch']:,})"
)
